In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/post_cell_13.pickle

In [ ]:
%%cudf.pandas.profile
### cell 14 ###

# total number of rows (remains on CPU but is cheap)
count = len(flights_df)

# Compute group sizes on GPU using a GPU‐accelerated count on a numeric column ('day')
df = (
    flights_df
    .groupby(["carrier", "flight", "dest"])["day"]
    .count()
    .reset_index(name="Size")
)

# Filter on the GPU
df_filtered = df[df["Size"] == 365]

# Iterate only over the (usually small) set of daily flights
for carrier, flight, dest in zip(
    df_filtered["carrier"],
    df_filtered["flight"],
    df_filtered["dest"]
):
    print(f"Carrier: {carrier}, Flight: {flight}, Destination: {dest}")

# Preserve the original loop index variable (last label in the RangeIndex)
i = len(df) - 1